# Modelagem Multinível do Efeito das Emendas PIX

Este notebook implementa a estratégia **multilevel step-up** descrita no TCC.
As etapas são:
1. **Modelo nulo**: estima a variância intra-partidos sem variáveis explanatórias.
2. **Modelo com interceptos aleatórios**: inclui `emendas_pix_per_capita_partido_prefeito_eleito` como efeito fixo, permitindo interceptos diferentes por partido.
3. **Modelo com interceptos e inclinações aleatórios**: além do intercepto, o efeito das emendas por habitante varia entre partidos.
4. **Modelo completo**: adiciona as variáveis *dummy* dos clusters socioeconômicos, controlando perfis municipais.


In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm # estimação de modelos
from scipy import stats # estatística chi2
from statsmodels.iolib.summary2 import summary_col # comparação entre modelos
from scipy.stats import gaussian_kde # inserção de KDEs em gráficos
from matplotlib.gridspec import GridSpec # plotagem de gráficos separados
import time # definição do intervalo de tempo entre gráficos com animação
import imageio # para geração de figura GIF
from tqdm import tqdm # adiciona um indicador de progresso do código


In [2]:
# Carrega a base unificada com dummies de clusters
base = pd.read_csv('../data/dados_com_clusters.csv')

# Remover 1s da coluna 'porcentual_votos_partido_prefeito_eleito'
base = base[base['porcentagem_votos_validos_2024'] < 1]

# Remover valores extremos da coluna 'emendas_pix_per_capita_partido_prefeito_eleito'
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

base = remove_outliers(base, 'emendas_pix_per_capita_partido_prefeito_eleito')

# Converte as colunas de cluster para inteiro (0/1)
for col in ['cluster_0', 'cluster_1', 'cluster_2', 'cluster_3']:
    base[col] = base[col].fillna(False).astype(int)


C:\Users\bruno\AppData\Local\Temp\ipykernel_13324\3404729404.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base[col] = base[col].fillna(False).astype(int)
C:\Users\bruno\AppData\Local\Temp\ipykernel_13324\3404729404.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base[col] = base[col].fillna(False).astype(int)
C:\Users\bruno\AppData\Local\Temp\ipykernel_13324\3404729404.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) inste

In [3]:
# Modelo nulo
null_model = smf.mixedlm(
    'porcentagem_votos_validos_2024 ~ 1',
    base,
    groups=base['sigla_partido_prefeito_eleito']
)
null_res = null_model.fit()
print(null_res.summary())


                   Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: porcentagem_votos_validos_2024
No. Observations: 4393    Method:             REML                          
No. Groups:       23      Scale:              0.0144                        
Min. group size:  2       Log-Likelihood:     inf                           
Max. group size:  727     Converged:          Yes                           
Mean group size:  191.0                                                     
------------------------------------------------------------------------------
             Coef.     Std.Err.      z      P>|z|      [0.025         0.975]  
------------------------------------------------------------------------------
Intercept    -0.117   567065.777   -0.000   1.000   -1111428.618   1111428.383
Group Var     0.000                                                           



e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\sit

In [4]:
# Modelo com interceptos aleatórios
ri_model = smf.mixedlm(
    'porcentagem_votos_validos_2024 ~ emendas_pix_per_capita_partido_prefeito_eleito',
    base,
    groups=base['sigla_partido_prefeito_eleito']
)
ri_res = ri_model.fit()
print(ri_res.summary())


                          Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     porcentagem_votos_validos_2024
No. Observations:      4393        Method:                 REML                          
No. Groups:            23          Scale:                  0.0142                        
Min. group size:       2           Log-Likelihood:         3091.9453                     
Max. group size:       727         Converged:              Yes                           
Mean group size:       191.0                                                             
-----------------------------------------------------------------------------------------
                                               Coef. Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------------
Intercept                                      0.578    0.004 144.570 0.000  0.570  0.586
emendas_pix_per_capita_partido_prefe

e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [5]:
# Modelo com interceptos e inclinações aleatórios
rs_model = smf.mixedlm(
    'porcentagem_votos_validos_2024 ~ emendas_pix_per_capita_partido_prefeito_eleito',
    base,
    groups=base['sigla_partido_prefeito_eleito'],
    re_formula='1 + emendas_pix_per_capita_partido_prefeito_eleito'
)
rs_res = rs_model.fit()
print(rs_res.summary())


e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(


                               Mixed Linear Model Regression Results
Model:                    MixedLM         Dependent Variable:         porcentagem_votos_validos_2024
No. Observations:         4393            Method:                     REML                          
No. Groups:               23              Scale:                      0.0141                        
Min. group size:          2               Log-Likelihood:             3028.4140                     
Max. group size:          727             Converged:                  No                            
Mean group size:          191.0                                                                     
----------------------------------------------------------------------------------------------------
                                                           Coef. Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------------------------------------
Intercept             

e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 100.828509
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix

In [6]:
# Modelo multinível completo com dummies dos clusters
full_model = smf.mixedlm(
    'porcentagem_votos_validos_2024 ~ emendas_pix_per_capita_partido_prefeito_eleito + cluster_0 + cluster_1 + cluster_2 + cluster_3',
    base,
    groups=base['sigla_partido_prefeito_eleito'],
    re_formula='1 + emendas_pix_per_capita_partido_prefeito_eleito'
)
full_res = full_model.fit()
print(full_res.summary())


e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(


                                Mixed Linear Model Regression Results
Model:                     MixedLM         Dependent Variable:         porcentagem_votos_validos_2024
No. Observations:          4393            Method:                     REML                          
No. Groups:                23              Scale:                      0.0139                        
Min. group size:           2               Log-Likelihood:             3038.7484                     
Max. group size:           727             Converged:                  No                            
Mean group size:           191.0                                                                     
-----------------------------------------------------------------------------------------------------
                                                           Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------------------------
Intercept   

e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 100.928070
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix

## Análise dos coeficientes
A seguir resumimos os parâmetros do modelo completo para verificar quais variáveis apresentam efeitos estatisticamente significativos.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
summary_df = pd.DataFrame({
    'coeficiente': full_res.params,
    'erro_padrao': full_res.bse,
    'p_valor': full_res.pvalues
})
summary_df

,coeficiente,erro_padrao,p_valor
Intercept,0.583805,0.028085,5.679267e-96
emendas_pix_per_capita_partido_prefeito_eleito,0.000620,0.004801,8.972493e-01
cluster_0,0.007309,0.026591,7.834283e-01
cluster_1,0.024312,0.037916,5.213952e-01
cluster_2,-0.017620,0.026532,5.066157e-01
cluster_3,0.022644,0.029851,4.481081e-01
Group Var,0.117060,NaN,NaN
Group x emendas_pix_per_capita_partido_prefeito_eleito Cov,0.000015,NaN,NaN
emendas_pix_per_capita_partido_prefeito_eleito Var,0.028298,NaN,NaN


In [8]:
from scipy import stats

# exemplo para intercepto aleatório
lr_stat = -2*(ri_res.llf - null_res.llf)
pval = stats.chi2.sf(lr_stat, df=1)
print(f"LR stat={lr_stat:.2f}, p-value={pval:.3f}")

from statsmodels.miscmodels import BetaModel
endog = base['porcentagem_votos_validos_2024']
exog = sm.add_constant(base[['emendas_pix_per_capita', 'idhm','pib_per_capita','densidade']])
beta_mod = BetaModel(endog, exog).fit()
print(beta_mod.summary())


LR stat=inf, p-value=0.000


ImportError: cannot import name 'BetaModel' from 'statsmodels.miscmodels' (e:\Learning\TCC-MBA\.venv\Lib\site-packages\statsmodels\miscmodels\__init__.py)

In [ ]:
import statsmodels.formula.api as smf
logit = smf.mixedlm("reeleito ~ emendas_pix_pc + idhm + pib_pc + densidade",
                    base, groups=base["sigla_partido_prefeito_eleito"],
                    family=sm.families.Binomial()).fit()
